# Model Evaluation and Prediction Examples

This notebook reviews saved V5 results and runs example predictions through the project predictor.

It uses the saved model file from:

```text
models/match_edge_ai_v5_calibrated_specialist_ensemble.pkl
```

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))

from match_edge_v5_predictor import MatchEdgeV5Predictor

DATA_DIR = ROOT / "data" / "processed"
MODEL_DIR = ROOT / "models"
REPORT_DIR = ROOT / "reports"

In [ ]:
metrics = pd.read_csv(REPORT_DIR / "model_metrics.csv")
weights = pd.read_csv(REPORT_DIR / "optimized_blend_weights.csv")

metrics

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(metrics["model"], metrics["log_loss"])
plt.title("Model comparison by log loss")
plt.xlabel("Model")
plt.ylabel("Log loss")
plt.xticks(rotation=80)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(weights["specialist"], weights["optimized_weight"])
plt.title("Optimized specialist blend weights")
plt.xlabel("Specialist")
plt.ylabel("Weight")
plt.xticks(rotation=70)
plt.tight_layout()
plt.show()

weights

In [ ]:
predictor = MatchEdgeV5Predictor(
    teams_csv=DATA_DIR / "wc2026_teams_model_input.csv",
    features_csv=DATA_DIR / "team_form_elo_latest.csv",
    model_path=MODEL_DIR / "match_edge_ai_v5_calibrated_specialist_ensemble.pkl",
)

predictor.valid_teams().head()

In [ ]:
result = predictor.predict(
    "United States",
    "Paraguay",
    venue_country="United States",
    team_a_rest_days=5,
    team_b_rest_days=4,
    team_a_travel_km=0,
    team_b_travel_km=1200,
    temperature_c=24,
    humidity_pct=55,
)

summary = {
    key: value
    for key, value in result.items()
    if key != "specialist_breakdown"
}

pd.Series(summary)

In [ ]:
breakdown = pd.DataFrame(result["specialist_breakdown"]).T
breakdown

In [ ]:
examples = [
    ("Mexico", "South Africa", "Mexico"),
    ("Canada", "Bosnia and Herzegovina", "Canada"),
    ("United States", "Paraguay", "United States"),
    ("Brazil", "Morocco", "Other"),
    ("Spain", "Uruguay", "Other"),
]

rows = []

for team_a, team_b, venue_country in examples:
    output = predictor.predict(team_a, team_b, venue_country=venue_country)
    rows.append({
        "match": f"{output['team_a_name']} vs {output['team_b_name']}",
        "team_a_win_probability": output["team_a_win_probability"],
        "draw_probability": output["draw_probability"],
        "team_b_win_probability": output["team_b_win_probability"],
        "predicted_result": output["predicted_result"],
        "expected_score": f"{output['team_a_expected_goals']} - {output['team_b_expected_goals']}",
    })

pd.DataFrame(rows)